# 07 — Cross-cell-type reachability transfer (K562 ↔ RPE1)

**Question.** The headline Th2→Th1 result rests on the geometry of a *convex cone* of
perturbation-effect vectors measured in CD4⁺ T cells. Is that geometry a property of the
**biology** (it would transfer to other cell types) or of the **specific cell type's measured
basis** (it would not)? This notebook answers that with two independent genome-scale
essential-gene screens in different human cell types.

**Data.** Replogle et al. 2022 K562 (leukemia) and RPE1 (retinal epithelium) essential-gene
CRISPRi Perturb-seq, obtained as harmonized files from the **CZI Virtual Cell Models**
`cz-benchmarks-data` bucket — the same platform that hosts this project's CD4⁺ T anchor.
These screens were previously unreachable (figshare-blocked); CZI resolves that.

**Design.** We restrict to the **843 single-gene perturbations present in both cell types**
and the **2,832 shared readout genes**, then build two row/column-aligned effect matrices
`E_K562` and `E_RPE1`. Each perturbation's effect is a pseudobulk logFC vs the non-targeting
control pool (no cross-perturbation centering — that would inject a rank-deficiency identity).
We then reuse the project's own `reachability.py` unchanged.

Three tests, increasing in stringency:
1. **Single-perturbation correspondence** — do matched knockdowns produce the same effect
   direction across cell types, above a shuffled-gene *and* a mismatched-perturbation null?
2. **Within- vs cross-cell-type reachability** — is a held-out target reachable from the cone
   of the *other* perturbations in its own cell type vs the other cell type?
3. **Minimal-recipe transfer** — even when a target is reachable in both, does the NNLS
   minimal recipe recruit the *same generators*?


In [ ]:
import sys, json, numpy as np
sys.path.insert(0, "..")                     # project root: reachability.py
import reachability as rr

# Effect-matrix checkpoint (built by build_effect_matrices.py from the two CZI h5ad files)
d = np.load("../analysis_cache/czi_data/cross_celltype_effects.npz", allow_pickle=True)
E_K562, E_RPE1 = d["E_K562"].astype(float), d["E_RPE1"].astype(float)
perts   = d["perturbations"].astype(str)
gene_id = d["gene_id"].astype(str)
gene_nm = d["gene_name"].astype(str)
P, G = E_K562.shape
print(f"E_K562 {E_K562.shape}  E_RPE1 {E_RPE1.shape}")
print(f"{P} shared single-gene perturbations x {G} shared readout genes")

## Sanity check — CRISPRi knocks down its own target

Before any cone reasoning: for every perturbation whose target gene is itself in the readout
space, the on-target self-effect must be **negative** (CRISPRi reduces the target's own mRNA).
This is the ground-truth orientation check.

In [ ]:
name2col = {g:i for i,g in enumerate(gene_nm)}
on_k, on_r = [], []
for pi, p in enumerate(perts):
    j = name2col.get(p)
    if j is not None:
        on_k.append(E_K562[pi, j]); on_r.append(E_RPE1[pi, j])
on_k, on_r = np.array(on_k), np.array(on_r)
print(f"on-target self-effect checked on {len(on_k)} perturbations")
print(f"  K562: mean {on_k.mean():+.3f}, fraction negative {np.mean(on_k<0):.1%}")
print(f"  RPE1: mean {on_r.mean():+.3f}, fraction negative {np.mean(on_r<0):.1%}")

## Test 1 — Single-perturbation effect correspondence

`cos(E_K562[i], E_RPE1[i])` for each shared perturbation `i`, against two nulls:
a **shuffled-gene** null (permute the gene axis — destroys all structure) and a
**mismatched-perturbation** null (compare gene *i* in K562 to a *different* gene in RPE1 —
this captures the shared essential-stress response that any knockdown induces).

In [ ]:
def cos(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(a @ b / (na * nb)) if na and nb else 0.0

rng = np.random.default_rng(0)
matched   = np.array([cos(E_K562[i], E_RPE1[i]) for i in range(P)])
null_shuf = np.array([cos(E_K562[i], E_RPE1[i][rng.permutation(G)]) for i in range(P)])
jj = rng.permutation(P); jj[jj==np.arange(P)] = (jj[jj==np.arange(P)]+1) % P
null_mm   = np.array([cos(E_K562[i], E_RPE1[jj[i]]) for i in range(P)])

print(f"matched cosine        median {np.median(matched):+.3f}")
print(f"shuffled-gene null    median {np.median(null_shuf):+.3f}  (95th {np.percentile(null_shuf,95):+.3f})")
print(f"mismatched-pert null  median {np.median(null_mm):+.3f}  (95th {np.percentile(null_mm,95):+.3f})")

The matched cosine (median **+0.35**) is far above the shuffled-gene null (≈0), but there is
a substantial **mismatched-perturbation** baseline (median +0.19): essential-gene knockdowns
share a common growth-arrest / stress direction regardless of which gene. To isolate the
*gene-specific* signal we (a) z-score each matched cosine against that perturbation's own
mismatched distribution, and (b) remove the common direction (mean effect) and re-measure.

In [ ]:
# (a) per-perturbation gene-specificity z
K = 200
spec_z = np.zeros(P); spec_p = np.zeros(P)
rng = np.random.default_rng(1)
for i in range(P):
    others = rng.choice(np.delete(np.arange(P), i), size=K, replace=False)
    mm = np.array([cos(E_K562[i], E_RPE1[o]) for o in others])
    spec_z[i] = (matched[i]-mm.mean())/mm.std() if mm.std() else 0.0
    spec_p[i] = (mm >= matched[i]).mean()

# (b) deflate common essential-stress direction
def deflate(E, c):
    cu = c/np.linalg.norm(c); return E - np.outer(E@cu, cu)
common_cos = cos(E_K562.mean(0), E_RPE1.mean(0))
Ek_d, Er_d = deflate(E_K562, E_K562.mean(0)), deflate(E_RPE1, E_RPE1.mean(0))
matched_defl = np.array([cos(Ek_d[i], Er_d[i]) for i in range(P)])

print(f"common essential-stress direction cosine: {common_cos:+.3f}")
print(f"gene-specific at p<0.05: {np.mean(spec_p<0.05):.1%};  z>2: {np.mean(spec_z>2):.1%}")
print(f"matched cosine after deflation: median {np.median(matched_defl):+.3f}; still positive for {np.mean(matched_defl>0):.1%}")

![Figure 1 — single-perturbation correspondence](../analysis_cache/czi_fig/nb07_fig1_correspondence.png)

**Figure 1.** (a) Matched same-gene knockdowns agree across cell types far above the
shuffled-gene null; a weaker mismatched-perturbation null reflects the shared essential-stress
response. (b) Gene-specific correspondence survives removing that common direction (median
+0.24; positive for 96 % of perturbations). (c) 39 % of perturbations are strongly
gene-specific (z>2) and 69 % reach p<0.05.

## Test 2 — Within- vs cross-cell-type reachability

For each held-out perturbation `i` we take its effect vector as the **target** `d` and ask
whether `d` lies in the non-negative cone of the *other* 842 perturbations:

- **within**: target `E_A[i]` vs cone of `E_A[others]` (same cell type — this is the ceiling)
- **cross**:  target `E_A[i]` vs cone of `E_B[others]` (other cell type's basis)

We report the cone residual (fraction of target norm unreachable) and the reachable cosine,
both directions. Runtime: 843 targets × 4 NNLS fits ≈ 5 min.

In [ ]:
idx = np.arange(P)
def fit(E_basis, dvec):
    r = rr.reachability(E_basis, dvec)
    return r.residual_norm, r.reachable_cosine

win_k_res=np.zeros(P); win_r_res=np.zeros(P); crs_kr_res=np.zeros(P); crs_rk_res=np.zeros(P)
win_k_cos=np.zeros(P); win_r_cos=np.zeros(P); crs_kr_cos=np.zeros(P); crs_rk_cos=np.zeros(P)
for i in idx:
    m = idx != i
    win_k_res[i], win_k_cos[i] = fit(E_K562[m], E_K562[i])   # K562 target, K562 basis
    win_r_res[i], win_r_cos[i] = fit(E_RPE1[m], E_RPE1[i])   # RPE1 target, RPE1 basis
    crs_kr_res[i], crs_kr_cos[i] = fit(E_RPE1[m], E_K562[i]) # K562 target, RPE1 basis
    crs_rk_res[i], crs_rk_cos[i] = fit(E_K562[m], E_RPE1[i]) # RPE1 target, K562 basis

print("median cone residual (lower = more reachable):")
print(f"  within K562 {np.median(win_k_res):.3f}   cross K562->RPE1 basis {np.median(crs_kr_res):.3f}")
print(f"  within RPE1 {np.median(win_r_res):.3f}   cross RPE1->K562 basis {np.median(crs_rk_res):.3f}")

To turn residuals into **verdicts** we calibrate against the project's shuffled-target null:
gene-permuted random targets fit to each basis give the reachable cosine achievable by chance.
A target is "reachable" if its reachable cosine exceeds the 95th percentile of that null.

In [ ]:
def null_band(E_basis, n=60, seed=0):
    rng = np.random.default_rng(seed); Pn = E_basis.shape[0]
    sel = rng.choice(Pn, n, replace=False); out=[]
    for i in sel:
        dp = E_basis[i][rng.permutation(E_basis.shape[1])]
        out.append(rr.reachability(np.delete(E_basis,i,0), dp).reachable_cosine)
    return np.array(out)

nb_k, nb_r = null_band(E_K562,60,0), null_band(E_RPE1,60,1)
thr_k, thr_r = np.percentile(nb_k,95), np.percentile(nb_r,95)
v_win_k, v_crs_kr = win_k_cos>thr_k, crs_kr_cos>thr_r   # K562 targets: within(K basis), cross(R basis)
v_win_r, v_crs_rk = win_r_cos>thr_r, crs_rk_cos>thr_k   # RPE1 targets
print(f"null 95th cosine: K562 basis {thr_k:.3f}, RPE1 basis {thr_r:.3f}")
print(f"reachable within/cross:  K562 tgt {v_win_k.mean():.0%}/{v_crs_kr.mean():.0%}   RPE1 tgt {v_win_r.mean():.0%}/{v_crs_rk.mean():.0%}")
print(f"verdict agreement:  K562 tgt {(v_win_k==v_crs_kr).mean():.1%}   RPE1 tgt {(v_win_r==v_crs_rk).mean():.1%}")

![Figure 2 — within vs cross reachability](../analysis_cache/czi_fig/nb07_fig2_reachability.png)

**Figure 2.** (a) Held-out targets are markedly **more reachable within their own cell type**
than across (residuals: K562 0.61 vs 0.87; RPE1 0.37 vs 0.68) — the cone geometry is partly
cell-type-specific. (b) Yet cross-type reachable cosines are correlated with within-type and
sit well above zero: the structure substantially transfers, just imperfectly. (c) Because the
843-generator non-negative cone is highly expressive, cross cosines clear the shuffled-target
null for essentially every target, so the **binary verdict agrees ~100 %** within vs cross —
the graded residual, not the verdict, carries the cell-type signal.

## Test 3 — Minimal-recipe transfer

The sharpest test. Even when a target is reachable in both cell types, does the NNLS
**minimal recipe** — the small set of top-weighted generator perturbations — recruit the same
genes? We compare top-10 supports by Jaccard overlap against a random-subset null.

- **same-target**: recipe for `E_K562[i]` fit in K562 vs recipe for `E_RPE1[i]` fit in RPE1
  (is the recipe to reach "gene *i*'s state" cell-type-invariant?)
- **cross-basis**: recipe for `E_K562[i]` fit in K562 vs the *same target* fit from RPE1's basis

In [ ]:
def support_global(E_basis, i, target_vec, k=10):
    m = idx != i; gg = idx[m]
    return set(gg[rr.reachability(E_basis[m], target_vec).support[:k]].tolist())

def jac(a,b): return len(a&b)/len(a|b) if (a|b) else 0.0
jac_q1=np.zeros(P); jac_q2k=np.zeros(P); jac_q2r=np.zeros(P)
for i in idx:
    sk = support_global(E_K562, i, E_K562[i])   # K562 target, native
    sr = support_global(E_RPE1, i, E_RPE1[i])   # RPE1 target, native
    skx = support_global(E_RPE1, i, E_K562[i])  # K562 target from RPE1 basis
    srx = support_global(E_K562, i, E_RPE1[i])  # RPE1 target from K562 basis
    jac_q1[i]=jac(sk,sr); jac_q2k[i]=jac(sk,skx); jac_q2r[i]=jac(sr,srx)

rng=np.random.default_rng(0); nn=[]
for _ in range(20000):
    a=set(rng.choice(P-1,10,replace=False)); b=set(rng.choice(P-1,10,replace=False)); nn.append(jac(a,b))
null95=np.percentile(nn,95)
print(f"random-null Jaccard: mean {np.mean(nn):.3f}, 95th {null95:.3f}")
print(f"same-target recipe overlap: median {np.median(jac_q1):.3f}  ({np.mean(jac_q1>null95):.0%} above null-95th)")
print(f"cross-basis recipe overlap: median {np.median(jac_q2k):.3f} (K562 tgt) / {np.median(jac_q2r):.3f} (RPE1 tgt)")

![Figure 3 — minimal-recipe transfer](../analysis_cache/czi_fig/nb07_fig3_recipe.png)

**Figure 3.** (a) Same-target recipes share components far above chance (median Jaccard 0.11 vs
random-null 0.006, ≈20×; 65 % exceed the null 95th percentile) — but the overlap is modest, so
recipes are **similar, not identical**, across cell types. (b) When the *same* target is reached
from the other cell type's basis, recipe overlap collapses to the null (0.05): the cone finds a
largely different generator set. (c) The transfer hierarchy — verdict ≫ effect direction ≫
minimal recipe. *(Panel c bars are separately-normalised summary scores — verdict agreement
fraction, median matched cosine, median same-target Jaccard — so their ranking is qualitative,
not a shared-scale comparison.)*

## Interpretation — what this says about the Th2→Th1 headline

**What transfers.** The *existence* of a reachable direction and the broad *effect geometry*
transfer across cell types. Matched knockdowns move both cell types in correlated directions
(median cosine +0.35, gene-specific for ~70 %), and held-out targets remain reachable from a
foreign-cell-type cone (cosine 0.50–0.73). The convex-cone framework is therefore **not an
artifact of one cell type** — the same machinery yields coherent, above-null structure in
leukemia and retinal-epithelial cells.

**What does not transfer.** The *specific minimal recipe* is substantially cell-type-specific.
Same-target recipes overlap only ~20 % of the way to identity, and recipes built from a foreign
basis share essentially nothing with the native one. Reaching a target state requires knowing
**that cell type's** perturbation responses.

**Implication for Th2→Th1.** This is a **robustness result with a sharp boundary condition**.
It supports the claim that the reachability *method* is sound and generalizes — a reviewer
asking "is the cone just a K562/T-cell curiosity?" is answered: no, the geometry is real in
three independent human cell types. But it also **bounds the headline's portability**: the
GATA3↓/TBX21↓-style minimal recipe the paper derives for Th2→Th1 is validated *for CD4⁺ T
cells*, and should **not** be assumed to be the recipe in another cell type without re-fitting
on that cell type's own perturbation basis. The direction transfers; the prescription does not.

**Caveats.**
- K562 and RPE1 are essential-gene **loss-of-function** screens; the T-cell anchor includes the
  full genome-wide library. The shared-perturbation basis here is 843 essential genes, enriched
  for core cellular machinery, which raises the common-stress baseline (Fig 1a) and inflates the
  raw cosine — hence the gene-specificity controls.
- The near-100 % binary verdict agreement reflects an **expressive over-complete cone**
  (843 non-negative generators, 2,832 genes), not perfect biological transfer. The graded
  residual and the recipe overlap are the honest discriminators.
- Effect magnitudes differ ~2× between cell types (RPE1 larger); all cross-type comparisons here
  are cosine/rank-based and therefore scale-invariant.


## Forward-note — transportability as a formal question (B4)

This notebook tests transfer *empirically*: does a cone learned in K562 carry to RPE1? The
answer — direction transfers, recipe does not — is exactly a **transportability** result in the
sense of Pearl & Bareinboim: an effect measured in a source population is valid in a target
population only if the two differ solely in ways that do not touch the causal pathway, a
condition made precise by a **selection diagram**.

The open case for the main project is the **aging axis**. The Th2→Th1 target is measured in the
same assay it is applied to, so transportability is close to automatic. The CD4 aging signature,
by contrast, is a target *direction* imported from a different study/context and pushed onto the
Perturb-seq effect basis — a genuine cross-population transport. The A1 sensitivity analysis
(`09_causal_validation_dossier.ipynb`) already flags the aging axes as fragile at Stim48hr (they
sit below the anisotropy null); B4 would say *why* in causal terms: without a selection diagram
that licenses the transport, the aging verdict is a source-population claim, not a target-population
one.

**B4** (a Future-Work item, prose only — see `../CAUSAL.md`) would draw the
selection diagram between the CRISPRi assay population and the aging-signature reference context,
identify which effects are transportable, and re-express the aging verdict as a transported —
rather than assumed-invariant — quantity.